In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import random
import glob
import scipy

In [ ]:
# # find all pickle files in the current directory using glob

# path = "/Users/ens/Library/Mobile Documents/com~apple~CloudDocs/Rupali/new"
# all_files = glob.glob(path + "/*.pkl")

# # create a list of dataframes
# li = []

# for filename in all_files:
#     df = pd.read_pickle(filename)
#     li.append(df)

# # concatenate the list of dataframes into one dataframe
# data = pd.concat(li, axis=0, ignore_index=True)

In [ ]:
# load datafram from pickle
data = pd.read_pickle('/Users/rupali/Documents/MARL/marl/data/diplomacy-v1-27k-msgs/output-2022-10-28_01-27-35_worker-0_data.pkl')
batch_size = 100

In [ ]:
data.head()

In [ ]:
def data_prep(data, batch_size, separated = False, ratios=[0.0, 0.8, 0.9]):

    # Shuffling by episode
    groups = [data for _, data in data.groupby('eps_id')]
    random.shuffle(groups)
    data = pd.concat(groups).reset_index(drop=True)
    data = data.sort_values(by=['eps_id','t','agent_index'],ignore_index=True)
    data = data.rename(columns={"action_dist_inputs": "logits"})
    data['probs'] = data['logits'].transform(scipy.special.softmax)

    # do train test validation split on the data
    # the data is split into ratios[0:1] train, ratios[1:2] test, ratios[2:]validation
    # the data is shuffled before splitting
    # splitting dataset by episodes
    num_episodes = len(data['eps_id'].unique())
    length_of_epi = max(data['t'].unique()) + 1
    num_agents = len(data['agent_index'].unique())
    _, train, test, val = np.split(data, [int(ratios[0]*length_of_epi*num_agents*num_episodes),int(ratios[1]*length_of_epi*num_agents*num_episodes), int(ratios[2]*length_of_epi*num_agents*num_episodes)])


    print("train shape: ", train.shape)
    print("test shape: ", test.shape)
    print("val shape: ", val.shape)


    # create test, train and validation dataloaders
    # the dataloaders will be used to train the neural network
    # the dataloaders will return a batch of observations and actions
    # the batch size is set to 100
    # the shuffle parameter is set to True so that the data is shuffled before each epoch

    all_train_dataloaders = []
    all_test_dataloaders = []
    all_val_dataloaders = []

    # 'separated' decides size of train, test and validation sets. 
    # If False, obs and actions are concatenated for separate agents. 
    # If True, obs and actions are separate for separate agents.
    if separated == True:
        for agent_ind in np.arange(num_agents):

            train_agent = train[train['agent_index'] == agent_ind]
            test_agent = test[test['agent_index'] == agent_ind]
            val_agent = val[val['agent_index'] == agent_ind]

            obs_train, obs_test, obs_val = np.array(train_agent['obs'].to_list()), np.array(test_agent['obs'].to_list()), np.array(val_agent['obs'].to_list())
            act_train, act_test, act_val = np.array(train_agent['actions'].to_list()), np.array(test_agent['actions'].to_list()), np.array(val_agent['actions'].to_list())
            ids_train, ids_test, ids_val = np.array(train_agent['agent_index'].to_list()), np.array(test_agent['agent_index'].to_list()), np.array(val_agent['agent_index'].to_list())
            logits_train, logits_test, logits_val = np.array(train_agent['logits'].to_list()), np.array(test_agent['logits'].to_list()), np.array(val_agent['logits'].to_list())
            probs_train, probs_test, probs_val = np.array(train_agent['probs'].to_list()), np.array(test_agent['probs'].to_list()), np.array(val_agent['probs'].to_list())


            train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_train), torch.from_numpy(act_train),torch.from_numpy(ids_train),torch.from_numpy(logits_train),torch.from_numpy(probs_train))
            train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            all_train_dataloaders.append(train_dataloader)

            test_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_test), torch.from_numpy(act_test),torch.from_numpy(ids_test),torch.from_numpy(logits_test),torch.from_numpy(probs_test))
            test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
            all_test_dataloaders.append(test_dataloader)

            val_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_val), torch.from_numpy(act_val),torch.from_numpy(ids_val),torch.from_numpy(logits_val),torch.from_numpy(probs_val))
            val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
            all_val_dataloaders.append(val_dataloader)
    else:
        obs_train, obs_test, obs_val = np.array(train['obs'].to_list()), np.array(test['obs'].to_list()), np.array(val['obs'].to_list())
        act_train, act_test, act_val = np.array(train['actions'].to_list()), np.array(test['actions'].to_list()), np.array(val['actions'].to_list())
        ids_train, ids_test, ids_val = np.array(train['agent_index'].to_list()), np.array(test['agent_index'].to_list()), np.array(val['agent_index'].to_list())
        logits_train, logits_test, logits_val = np.array(train['logits'].to_list()), np.array(test['logits'].to_list()), np.array(val['logits'].to_list())
        probs_train, probs_test, probs_val = np.array(train['probs'].to_list()), np.array(test['probs'].to_list()), np.array(val['probs'].to_list())


        train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_train), torch.from_numpy(act_train),torch.from_numpy(ids_train),torch.from_numpy(logits_train),torch.from_numpy(probs_train))
        all_train_dataloaders = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        test_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_test), torch.from_numpy(act_test),torch.from_numpy(ids_test),torch.from_numpy(logits_test),torch.from_numpy(probs_test))
        all_test_dataloaders = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

        val_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_val), torch.from_numpy(act_val),torch.from_numpy(ids_val),torch.from_numpy(logits_val),torch.from_numpy(probs_val))
        all_val_dataloaders = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)


    return all_train_dataloaders, all_test_dataloaders, all_val_dataloaders

In [ ]:
# get dataloaders for train, test and validation data
all_train_dataloaders, all_test_dataloaders, all_val_dataloaders = data_prep(data, batch_size, separated=False)